## Project Goal

**Project Name:** Real-Time Data Drift Detector

**Goal:** Detect when incoming data distribution is different from training data.

This notebook will guide you through a step-by-step implementation of a real-time data drift detector using `pandas`, `numpy`, `scikit-learn`, and `joblib`.

---

\## Load & Train Model

This step focuses on creating a baseline by training a machine learning model on a static dataset. This 'old data' will serve as our reference distribution for detecting future data drift.

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib
import os

### Generate or load a sample dataset and convert to pandas DataFrame

We'll use the Breast Cancer dataset from `sklearn`, which is a good example of a tabular dataset for classification. Converting it to a pandas DataFrame provides a more convenient structure for data manipulation.

In [2]:
# Load the Breast Cancer dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

df = pd.concat([X, y], axis=1)

print("Dataset head:")
display(df.head())
print(f"Dataset shape: {df.shape}")

Dataset head:


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


Dataset shape: (569, 31)
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


### Split into train/test

Splitting the dataset into training and testing sets is crucial for evaluating the model's performance on unseen data and preventing overfitting.

In [3]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Training data shape: (455, 30)
Testing data shape: (114, 30)


### Train a RandomForestClassifier

A RandomForestClassifier is an ensemble learning method suitable for classification tasks. It builds multiple decision trees and merges them to get a more accurate and stable prediction.

In [4]:
# Train a RandomForestClassifier
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Predict on the test set and calculate accuracy
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.4f}")

Model Accuracy: 0.9649


### Save trained model and training dataset

Saving the trained model (`model.pkl`) and the training dataset (`train.csv`) allows us to reload them later without retraining. The `train.csv` will be our reference for drift detection.

In [5]:
# Create a directory for artifacts if it doesn't exist
output_dir = 'artifacts'
os.makedirs(output_dir, exist_ok=True)

# Save the trained model
model_path = os.path.join(output_dir, 'model.pkl')
joblib.dump(model, model_path)
print(f"Trained model saved to {model_path}")

# Save the training dataset (including target variable)
train_df = pd.concat([X_train, y_train], axis=1)
train_data_path = os.path.join(output_dir, 'train.csv')
train_df.to_csv(train_data_path, index=False)
print(f"Training data saved to {train_data_path}")

Trained model saved to artifacts/model.pkl
Training data saved to artifacts/train.csv


## Create New (Drifted) Data

This step focuses on simulating real-world scenarios where the incoming data distribution might differ from the training data. We'll intentionally introduce drift to later test our detection mechanisms.

### Load `train.csv`

We need to load the `train.csv` file, which contains our baseline training data, to use as a reference for creating the 'drifted' new data.

In [6]:
# Load the training data (our reference data)
train_df = pd.read_csv('artifacts/train.csv')

print("Loaded training data head:")
display(train_df.head())
print(f"Loaded training data shape: {train_df.shape}")

Loaded training data head:


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,9.029,17.33,58.79,250.5,0.10660,0.14130,0.31300,0.04375,0.2111,0.08046,...,22.65,65.50,324.7,0.14820,0.43650,1.25200,0.17500,0.4228,0.11750,1
1,21.090,26.57,142.70,1311.0,0.11410,0.28320,0.24870,0.14960,0.2395,0.07398,...,33.48,176.50,2089.0,0.14910,0.75840,0.67800,0.29030,0.4098,0.12840,0
2,9.173,13.86,59.20,260.9,0.07721,0.08751,0.05988,0.02180,0.2341,0.06963,...,19.23,65.59,310.1,0.09836,0.16780,0.13970,0.05087,0.3282,0.08490,1
3,10.650,25.22,68.01,347.0,0.09657,0.07234,0.02379,0.01615,0.1897,0.06329,...,35.19,77.98,455.7,0.14990,0.13980,0.11250,0.06136,0.3409,0.08147,1
4,10.170,14.88,64.55,311.9,0.11340,0.08061,0.01084,0.01290,0.2743,0.06960,...,17.45,69.86,368.6,0.12750,0.09866,0.02168,0.02579,0.3557,0.08020,1


Loaded training data shape: (455, 31)


### Create a modified version of dataset to simulate drift

To simulate data drift, we will modify a portion of the original training data. We'll achieve this by:
1. **Selecting a subset of features**: Focus on a few key features to introduce changes.
2. **Adding random noise**: Introduce random fluctuations to some feature values to shift their distribution.
3. **Scaling values**: Multiply some features by a factor to change their scale.
4. **Shifting the mean**: Add a constant value to some features to shift their central tendency.

This intentional modification allows us to create a dataset (`new_data.csv`) that we know contains drift, which is essential for testing our drift detection logic later.

In [7]:
# Create a copy of the training data to simulate new incoming data
new_data_df = train_df.copy()

# Introduce drift in selected features

# Example 1: Shift the mean for 'mean radius'
new_data_df['mean radius'] = new_data_df['mean radius'] * 1.1 + np.random.normal(0, 0.5, len(new_data_df))

# Example 2: Add more noise to 'mean texture'
new_data_df['mean texture'] = new_data_df['mean texture'] + np.random.normal(0, 2, len(new_data_df))

# Example 3: Scale 'mean perimeter' and add a slight shift
new_data_df['mean perimeter'] = new_data_df['mean perimeter'] * 0.95 + 5 + np.random.normal(0, 1, len(new_data_df))

print("Drifted new data head:")
display(new_data_df.head())
print(f"Drifted new data shape: {new_data_df.shape}")

Drifted new data head:


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,10.097116,18.735139,61.421677,250.5,0.10660,0.14130,0.31300,0.04375,0.2111,0.08046,...,22.65,65.50,324.7,0.14820,0.43650,1.25200,0.17500,0.4228,0.11750,1
1,23.193706,22.596035,139.275176,1311.0,0.11410,0.28320,0.24870,0.14960,0.2395,0.07398,...,33.48,176.50,2089.0,0.14910,0.75840,0.67800,0.29030,0.4098,0.12840,0
2,9.613995,17.236923,61.505171,260.9,0.07721,0.08751,0.05988,0.02180,0.2341,0.06963,...,19.23,65.59,310.1,0.09836,0.16780,0.13970,0.05087,0.3282,0.08490,1
3,13.247695,24.652422,69.149446,347.0,0.09657,0.07234,0.02379,0.01615,0.1897,0.06329,...,35.19,77.98,455.7,0.14990,0.13980,0.11250,0.06136,0.3409,0.08147,1
4,10.978568,17.744935,65.588350,311.9,0.11340,0.08061,0.01084,0.01290,0.2743,0.06960,...,17.45,69.86,368.6,0.12750,0.09866,0.02168,0.02579,0.3557,0.08020,1


Drifted new data shape: (455, 31)


### Save as `new_data.csv`

Saving this 'drifted' dataset ensures we have a consistent file to work with for subsequent steps, representing new data arriving over time.

In [8]:
# Save the new, 'drifted' dataset
new_data_path = os.path.join(output_dir, 'new_data.csv')
new_data_df.to_csv(new_data_path, index=False)
print(f"New (drifted) data saved to {new_data_path}")

New (drifted) data saved to artifacts/new_data.csv


## Implement PSI (CORE PART)

Population Stability Index (PSI) is a widely used metric to quantify how much a population has shifted over time. It compares the distribution of a variable in a new dataset to its distribution in a baseline dataset (our training data).

### Explanation of PSI

In simple terms, PSI measures the difference in the distribution of a feature between two datasets. It works by:
1.  **Binning**: Dividing the range of the feature into a fixed number of bins (e.g., 10).
2.  **Calculating Proportions**: For each bin, it calculates the percentage of data points falling into that bin for both the baseline (training) data and the new data.
3.  **Calculating Difference**: It finds the difference between these percentages for each bin.
4.  **Weighting**: It then weights this difference by taking the natural logarithm of the ratio of the new data proportion to the baseline proportion.
5.  **Summing**: Finally, it sums these weighted differences across all bins to get the total PSI value.


**Interpretation of PSI values:**
*   **< 0.1**: No significant shift / No Drift
*   **0.1 - 0.25**: Moderate shift / Moderate Drift
*   **> 0.25**: Significant shift / High Drift

This provides a single score to understand how much a feature's distribution has changed.

### Write a function to calculate PSI

This function will take two pandas Series (representing a single column from the reference and new datasets) and calculate the PSI. We'll use 10 bins and add a small epsilon to avoid division by zero errors.

In [9]:
def calculate_psi(baseline_series, new_series, num_bins=10):
    """
    Calculate the Population Stability Index (PSI) for a single feature.

    Args:
        baseline_series (pd.Series): The series from the baseline (training) dataset.
        new_series (pd.Series): The series from the new (current) dataset.
        num_bins (int): The number of bins to use for bucketing.

    Returns:
        float: The calculated PSI value.
    """
    # Combine the series to determine the overall min and max for consistent binning
    all_data = pd.concat([baseline_series, new_series])
    min_val = all_data.min()
    max_val = all_data.max()

    # Create bins - ensures consistent bin edges across both series
    bins = np.linspace(min_val, max_val, num_bins + 1)

    # Calculate counts for each bin in baseline and new data
    baseline_counts = np.histogram(baseline_series, bins=bins)[0]
    new_counts = np.histogram(new_series, bins=bins)[0]

    # Calculate proportions, add a small epsilon to avoid division by zero
    epsilon = 1e-6

    baseline_proportions = (baseline_counts / len(baseline_series)) + epsilon
    new_proportions = (new_counts / len(new_series)) + epsilon

    # Calculate PSI for each bin
    psi_per_bin = (new_proportions - baseline_proportions) * np.log(new_proportions / baseline_proportions)

    # Sum up PSI for all bins
    total_psi = psi_per_bin.sum()

    return total_psi

print("PSI function defined.")

PSI function defined.


### Apply PSI for all numerical columns and store results

Now, we'll iterate through all numerical features present in both our training data (`train_df`) and the newly generated data (`new_data_df`) to calculate their respective PSI values. The results will be stored in a DataFrame for easy analysis.

In [10]:
# Load the training data and new data (if not already loaded)
try:
    train_df = pd.read_csv(os.path.join(output_dir, 'train.csv'))
    new_data_df = pd.read_csv(os.path.join(output_dir, 'new_data.csv'))
except FileNotFoundError:
    print("Error: 'train.csv' or 'new_data.csv' not found. Please ensure previous steps were executed.")
    exit()

# Identify numerical columns common to both dataframes, excluding the target variable
numerical_cols = train_df.select_dtypes(include=np.number).columns.tolist()
if 'target' in numerical_cols:
    numerical_cols.remove('target')

psi_results = []

for col in numerical_cols:
    psi_value = calculate_psi(train_df[col], new_data_df[col])
    psi_results.append({'Feature': col, 'PSI_Value': psi_value})

psi_df = pd.DataFrame(psi_results)

print("PSI values calculated for all numerical features:")
display(psi_df.sort_values(by='PSI_Value', ascending=False).head())

PSI values calculated for all numerical features:


,Feature,PSI_Value
0,mean radius,0.303592
1,mean texture,0.059027
2,mean perimeter,0.016815
3,mean area,0.000000
4,mean smoothness,0.000000


## Drift Detection Logic

With PSI values calculated, we now need a way to interpret them. This step defines thresholds to categorize drift severity and then applies this logic to each feature, culminating in an overall drift summary.

### Use PSI thresholds and print feature-wise drift status

We will apply the standard PSI thresholds to classify each feature's drift status. The results for each feature, including its name, PSI value, and drift status, will be printed in a clear format.

In [11]:
# Define PSI thresholds
PSI_THRESHOLD_NO_DRIFT = 0.1
PSI_THRESHOLD_MODERATE_DRIFT = 0.25

def get_drift_status(psi_value):
    if psi_value < PSI_THRESHOLD_NO_DRIFT:
        return "No Drift"
    elif PSI_THRESHOLD_NO_DRIFT <= psi_value < PSI_THRESHOLD_MODERATE_DRIFT:
        return "Moderate Drift"
    else:
        return "High Drift"

print("\n--- Feature-wise Drift Status ---")
overall_drift_status = "No overall drift detected."
high_drift_features = []
moderate_drift_features = []

for index, row in psi_df.iterrows():
    feature = row['Feature']
    psi_value = row['PSI_Value']
    status = get_drift_status(psi_value)
    print(f"Feature: {feature:<25} | PSI: {psi_value:.4f} | Status: {status}")

    if status == "High Drift":
        high_drift_features.append(feature)
        overall_drift_status = "High overall drift detected!"
    elif status == "Moderate Drift" and "High overall drift detected!" not in overall_drift_status:
        moderate_drift_features.append(feature)
        if "Moderate overall drift detected!" not in overall_drift_status:
            overall_drift_status = "Moderate overall drift detected!"

# Display the PSI DataFrame with drift status
psi_df['Drift_Status'] = psi_df['PSI_Value'].apply(get_drift_status)
print("\nPSI Results with Drift Status:")
display(psi_df.sort_values(by='PSI_Value', ascending=False))



--- Feature-wise Drift Status ---
Feature: mean radius               | PSI: 0.3036 | Status: High Drift
Feature: mean texture              | PSI: 0.0590 | Status: No Drift
Feature: mean perimeter            | PSI: 0.0168 | Status: No Drift
Feature: mean area                 | PSI: 0.0000 | Status: No Drift
Feature: mean smoothness           | PSI: 0.0000 | Status: No Drift
Feature: mean compactness          | PSI: 0.0000 | Status: No Drift
Feature: mean concavity            | PSI: 0.0000 | Status: No Drift
Feature: mean concave points       | PSI: 0.0000 | Status: No Drift
Feature: mean symmetry             | PSI: 0.0000 | Status: No Drift
Feature: mean fractal dimension    | PSI: 0.0000 | Status: No Drift
Feature: radius error              | PSI: 0.0000 | Status: No Drift
Feature: texture error             | PSI: 0.0000 | Status: No Drift
Feature: perimeter error           | PSI: 0.0000 | Status: No Drift
Feature: area error                | PSI: 0.0000 | Status: No Drift
Feature: sm

,Feature,PSI_Value,Drift_Status
0,mean radius,0.303592,High Drift
1,mean texture,0.059027,No Drift
2,mean perimeter,0.016815,No Drift
3,mean area,0.000000,No Drift
4,mean smoothness,0.000000,No Drift
5,mean compactness,0.000000,No Drift
6,mean concavity,0.000000,No Drift
7,mean concave points,0.000000,No Drift
8,mean symmetry,0.000000,No Drift
9,mean fractal dimension,0.000000,No Drift


### Print overall drift summary

1.   List item
2.   List item



After evaluating each feature, an overall summary provides a concise overview of the data's stability, indicating if there's no, moderate, or high drift across the dataset.

In [12]:
print("\n--- Overall Drift Summary ---")
print(overall_drift_status)

if high_drift_features:
    print(f"Features with High Drift: {', '.join(high_drift_features)}")
elif moderate_drift_features:
    print(f"Features with Moderate Drift: {', '.join(moderate_drift_features)}")



--- Overall Drift Summary ---
High overall drift detected!
Features with High Drift: mean radius


## Simulate Real-Time Monitoring

This step simulates a real-world scenario where new data arrives continuously. We will process this data in batches and apply our PSI-based drift detection to each batch against the original training data. This demonstrates how the system can monitor for drift over time.

### Explanation: Why batching?

In real-time data processing, it's often impractical and inefficient to analyze every single data point individually. Instead, data is typically collected and processed in small groups or 'batches' over a short period. This approach offers several advantages:

*   **Efficiency**: Processing multiple records at once can be more computationally efficient than processing them one by one.
*   **Resource Management**: It allows for better management of computational resources, as the system can allocate resources for processing batches rather than constantly reacting to individual data points.
*   **Statistical Stability**: For drift detection methods like PSI, which rely on distributions, having a sufficient number of data points in a batch provides more statistically stable and meaningful results than trying to assess drift on very small samples.
*   **Latency vs. Throughput**: Batching balances the need for timely detection (low latency) with the ability to process large volumes of data (high throughput).

### Load new_data.csv, process in batches, calculate PSI, print status, and wait

We'll load the `new_data.csv` and split it into chunks to simulate incoming data. For each chunk, we'll calculate the PSI for each feature against the `train_df` (our baseline) and report any detected drift. A `time.sleep()` is used to mimic the delay between incoming data batches.

In [15]:
import time

# Load the new (drifted) data
new_data_full = pd.read_csv(os.path.join(output_dir, 'new_data.csv'))

# Load the training data (our reference data)
train_df = pd.read_csv(os.path.join(output_dir, 'train.csv'))

# Identify numerical columns common to both dataframes, excluding the target variable
numerical_cols = train_df.select_dtypes(include=np.number).columns.tolist()
if 'target' in numerical_cols:
    numerical_cols.remove('target')

# Define batch size
batch_size = 100
num_batches = (len(new_data_full) + batch_size - 1) // batch_size

print("\n--- Simulating Real-Time Monitoring ---")

for i in range(num_batches):
    start_idx = i * batch_size
    end_idx = min((i + 1) * batch_size, len(new_data_full))
    current_batch_df = new_data_full.iloc[start_idx:end_idx]

    print(f"\nProcessing batch {i+1}/{num_batches} (rows {start_idx} to {end_idx-1})...")

    batch_psi_results = []
    high_drift_in_batch = []
    moderate_drift_in_batch = []

    for col in numerical_cols:
        if col in current_batch_df.columns and col in train_df.columns:
            # Ensure the batch has enough data to calculate meaningful PSI
            if len(current_batch_df[col].dropna()) > 1 and len(train_df[col].dropna()) > 1:
                psi_value = calculate_psi(train_df[col], current_batch_df[col])
                status = get_drift_status(psi_value)
                batch_psi_results.append({'Feature': col, 'PSI_Value': psi_value, 'Status': status})
                if status == "High Drift":
                    high_drift_in_batch.append(col)
                elif status == "Moderate Drift":
                    moderate_drift_in_batch.append(col)
            else:
                batch_psi_results.append({'Feature': col, 'PSI_Value': np.nan, 'Status': "Insufficient data"})
        else:
            batch_psi_results.append({'Feature': col, 'PSI_Value': np.nan, 'Status': "Feature not found"})

    batch_psi_df = pd.DataFrame(batch_psi_results)

    if high_drift_in_batch:
        print(f"High Drift Detected in Batch {i+1} for features: {', '.join(high_drift_in_batch)}")
    elif moderate_drift_in_batch:
        print(f"Moderate Drift Detected in Batch {i+1} for features: {', '.join(moderate_drift_in_batch)}")
    else:
        print(f"No significant drift detected in Batch {i+1}.")

    # Display top 5 features by PSI for this batch
    display(batch_psi_df.sort_values(by='PSI_Value', ascending=False).head())

    # Simulate real-time delay
    time.sleep(2)

print("\n--- Real-Time Monitoring Simulation Complete ---")


--- Simulating Real-Time Monitoring ---

Processing batch 1/5 (rows 0 to 99)...
High Drift Detected in Batch 1 for features: mean radius, fractal dimension error, worst compactness


,Feature,PSI_Value,Status
25,worst compactness,0.416025,High Drift
0,mean radius,0.288883,High Drift
19,fractal dimension error,0.271985,High Drift
3,mean area,0.249169,Moderate Drift
2,mean perimeter,0.226867,Moderate Drift



Processing batch 2/5 (rows 100 to 199)...
High Drift Detected in Batch 2 for features: mean radius, mean perimeter, concave points error


,Feature,PSI_Value,Status
0,mean radius,0.369458,High Drift
2,mean perimeter,0.293721,High Drift
17,concave points error,0.270512,High Drift
9,mean fractal dimension,0.247448,Moderate Drift
6,mean concavity,0.243028,Moderate Drift



Processing batch 3/5 (rows 200 to 299)...
High Drift Detected in Batch 3 for features: mean radius


...


Processing batch 4/5 (rows 300 to 399)...
High Drift Detected in Batch 4 for features: mean radius, mean area, texture error, symmetry error, worst texture


,Feature,PSI_Value,Status
0,mean radius,0.532068,High Drift
18,symmetry error,0.495428,High Drift
11,texture error,0.384963,High Drift
21,worst texture,0.379661,High Drift
3,mean area,0.264744,High Drift



Processing batch 5/5 (rows 400 to 454)...
High Drift Detected in Batch 5 for features: mean radius, mean texture, mean smoothness, mean concave points, mean symmetry, mean fractal dimension, texture error, smoothness error, compactness error, concavity error, concave points error, symmetry error, worst texture, worst area, worst smoothness, worst compactness, worst concave points


,Feature,PSI_Value,Status
0,mean radius,1.238646,High Drift
9,mean fractal dimension,0.754567,High Drift
1,mean texture,0.689416,High Drift
16,concavity error,0.600706,High Drift
25,worst compactness,0.533890,High Drift



--- Real-Time Monitoring Simulation Complete ---
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


## Simple Dashboard

This final step involves creating a simple dashboard to visualize the PSI values and drift statuses. While a full-fledged Streamlit or Dash app would be ideal, for this notebook, we'll create a basic interactive display using `ipywidgets` and `matplotlib` that can be refreshed.

### Show PSI values & drift status, use table + visualization, add refresh button

We'll create a function that displays the PSI results, highlighting features with drift. This function will be called by an `ipywidgets.Button` to simulate a refresh capability, allowing you to re-run the drift analysis (or, in a real scenario, fetch new data and re-evaluate).

In [14]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Ensure train_df and psi_df are available
try:
    train_df = pd.read_csv(os.path.join(output_dir, 'train.csv'))
    new_data_df = pd.read_csv(os.path.join(output_dir, 'new_data.csv'))
except FileNotFoundError:
    print("Error: 'train.csv' or 'new_data.csv' not found. Please ensure previous steps were executed.")
    # Fallback to previously computed psi_df if files are not found but psi_df exists
    if 'psi_df' not in locals() and 'psi_df' not in globals():
        print("Exiting as psi_df is also not available.")
        exit()

def run_drift_analysis():
    # Recalculate PSI values (simulating a fresh run)
    numerical_cols = train_df.select_dtypes(include=np.number).columns.tolist()
    if 'target' in numerical_cols:
        numerical_cols.remove('target')

    psi_results = []
    for col in numerical_cols:
        psi_value = calculate_psi(train_df[col], new_data_df[col])
        psi_results.append({'Feature': col, 'PSI_Value': psi_value})

    current_psi_df = pd.DataFrame(psi_results)
    current_psi_df['Drift_Status'] = current_psi_df['PSI_Value'].apply(get_drift_status)

    # Sort for better visualization
    current_psi_df = current_psi_df.sort_values(by='PSI_Value', ascending=False).reset_index(drop=True)

    clear_output(wait=True)
    print("--- Data Drift Dashboard (Refreshed) ---")

    # Display table
    display(current_psi_df)

    # Simple visualization: Bar plot of top 10 PSI values
    plt.figure(figsize=(12, 6))
    top_features = current_psi_df.head(10)
    colors = ['red' if status == 'High Drift' else 'orange' if status == 'Moderate Drift' else 'green' for status in top_features['Drift_Status']]
    plt.bar(top_features['Feature'], top_features['PSI_Value'], color=colors)
    plt.ylabel('PSI Value')
    plt.title('Top 10 Features by PSI Value')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    # Overall Summary
    overall_status_summary = "No overall drift detected."
    if 'High Drift' in current_psi_df['Drift_Status'].values:
        overall_status_summary = "High overall drift detected!"
    elif 'Moderate Drift' in current_psi_df['Drift_Status'].values:
        overall_status_summary = "Moderate overall drift detected!"

    print(f"\nOverall Drift Status: {overall_status_summary}")

# Create a button widget
refresh_button = widgets.Button(description="Refresh Drift Analysis")
output = widgets.Output()

def on_button_click(b):
    with output:
        run_drift_analysis()

refresh_button.on_click(on_button_click)

# Display the button and initial analysis
display(refresh_button, output)
with output:
    run_drift_analysis()


Button(description='Refresh Drift Analysis', style=ButtonStyle())

Output()